## Persisting Flow State
.
One of CrewAI’s most powerful features is the ability to persist flow state across executions. This enables workflows that can be paused, resumed, and even recovered after failures
​
## The @persist() Decorator

The @persist() decorator automates state persistence, saving your flow’s state at key points in execution.
​
1.Class-Level Persistence

When applied at the class level, @persist() saves state after every method execution

## Why persistence matters here

Because of class-level @persist():

State is saved after every method

If the program stops after generate_draft

The flow can resume from human_review

In [12]:
from crewai.flow.flow import Flow, listen, start
from crewai.flow.persistence import persist
from pydantic import BaseModel

In [13]:
class CounterState(BaseModel): # structured state 
    value: int = 0

In [ ]:
@persist()  # Apply to the entire flow class
class PersistentCounterFlow(Flow[CounterState]):
    @start() # step_1
    def increment(self):
        self.state.value += 1  # 2+1 = 3
        print(f"Incremented to {self.state.value}")
        return self.state.value

    @listen(increment) # step_2
    def double(self, value):
        self.state.value = value * 2
        print(f"Doubled to {self.state.value}")
        return self.state.value



In [15]:
# First run
flow1 = PersistentCounterFlow()
result1 = flow1.kickoff()
print(f"First run result: {result1}")

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: PersistentCounterFlow                                                                                    │
│  ID: 643e4997-2b3d-4343-8347-5ed9270f0028                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: PersistentCounterFlow                                                                                    │
│  ID: 643e4997-2b3d-4343-8347-5ed9270f0028                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: 643e4997-2b3d-4343-8347-5ed9270f0028

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: increment                                                                                              │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Incremented to 1


...

Doubled to 2


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: double                                                                                                 │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: double                                                                                                 │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: PersistentCounterFlow                                                                                    │
│  ID: 643e4997-2b3d-4343-8347-5ed9270f0028                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

First run result: 2


In [16]:
print(flow1.state)

StateWithId(value=2, id='643e4997-2b3d-4343-8347-5ed9270f0028')


In [19]:
result2= flow1.kickoff() # 2nd time calling the same flow using the same object name
print(f"First run result: {result2}")

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: PersistentCounterFlow                                                                                    │
│  ID: 643e4997-2b3d-4343-8347-5ed9270f0028                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: PersistentCounterFlow                                                                                    │
│  ID: 643e4997-2b3d-4343-8347-5ed9270f0028                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: 643e4997-2b3d-4343-8347-5ed9270f0028

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: increment                                                                                              │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Incremented to 15


...

Doubled to 30


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: double                                                                                                 │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: double                                                                                                 │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: PersistentCounterFlow                                                                                    │
│  ID: 643e4997-2b3d-4343-8347-5ed9270f0028                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

First run result: 30


## One simple sentence

Persistence allows the flow to remember previous state values across executions.

## Method-Level Persistence

For more granular control, you can apply @persist() to specific methods:

In [20]:
from crewai.flow.flow import Flow, listen, start
from crewai.flow.persistence import persist


In [21]:
class SelectivePersistFlow(Flow):
    @start()
    def first_step(self):
        self.state["count"] = 1
        return "First step"

    @persist()  # Only persist after this method
    @listen(first_step)
    def important_step(self, prev_result):
        self.state["count"] += 1
        self.state["important_data"] = "This will be persisted"
        return "Important step completed"

    @listen(important_step)
    def final_step(self, prev_result):
        self.state["count"] += 1
        return f"Complete with count {self.state['count']}"

In [22]:
method_level_persistant = SelectivePersistFlow()
print(method_level_persistant.kickoff())

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SelectivePersistFlow                                                                                     │
│  ID: 8339fdc7-6647-437d-bced-2394870fc006                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: SelectivePersistFlow                                                                                     │
│  ID: 8339fdc7-6647-437d-bced-2394870fc006                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: 8339fdc7-6647-437d-bced-2394870fc006

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: first_step                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: first_step                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: important_step                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: final_step                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: final_step                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: SelectivePersistFlow                                                                                     │
│  ID: 8339fdc7-6647-437d-bced-2394870fc006                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Complete with count 3


## 1️⃣ HITL Concept

HITL (Human-in-the-Loop) means:

AI performs a step

Workflow pauses

Human reviews / gives feedback

Workflow continues

In real applications:

AI Step → Send data to UI → Human reviews → UI sends response → Flow continues

Example:

Generate email → UI shows email → Human approves → Email sent

## 2️⃣ Why Persistence is Needed for HITL

When the flow waits for human input, the system might wait:

5 seconds

5 minutes

5 hours

even days

During this waiting time, we must save the workflow state.

That is why we use persistence.

So:


AI step finished

↓

State saved (persisted)

↓

Waiting for human feedback

↓

Human sends response

↓

Flow resumes

What happens during HITL

When the flow reaches a human review step, the system pauses and waits for the user.

Example:

Step1 → AI generates response

Step2 → Wait for human approval (HITL)

Step3 → Continue workflow

Now imagine the user takes 5 minutes to respond.

During those 5 minutes:

Server may restart

Process may crash

Session may expire

Memory may clear

If persistence is NOT used:

❌ The intermediate state can be lost

❌ Workflow must restart from the beginning

With Persistence

When persistence is enabled:

Step1 → AI generates response

State saved

↓

Step2 → Waiting for human input

↓

User responds after 5 minutes

↓

Workflow resumes safely

Even if:

server restarts

process crashes

session closes

CrewAI can reload the saved state and continue.